# R2R Annotation Trajectory Parser
Extract the trajectory ID and step count for each data point in annotations.json.

In [ ]:
# Cell 1 — Imports & Config
import ijson
import pandas as pd

ANNOTATIONS_PATH = "/home/djonna1/scratchtinoosh/iros_dataset/Qwen-Dataset/test/annotations.json"
OUTPUT_PATH = "/home/djonna1/scratchtinoosh/iros_dataset/Qwen-Dataset/test/annotation_trajectory_info.csv"

In [2]:
# Cell 2 — Stream-parse annotations.json and extract trajectory ID and step count
# video_id format: "{trajectory_id}-{num_steps}"
# rsplit from the right to handle trajectory IDs that may contain "-"
records = []

with open(ANNOTATIONS_PATH, "rb") as f:
    for item in ijson.items(f, "item"):
        video_id = item["video_id"]
        trajectory_id, num_steps = video_id.rsplit("-", 1)
        records.append({
            "video_id": video_id,
            "trajectory_id": trajectory_id,
            "num_steps": int(num_steps)
        })

print(f"Parsed {len(records)} data points")

Parsed 353894 data points


In [3]:
# Cell 3 — Build DataFrame and print basic statistics
df = pd.DataFrame(records)

print(f"Total data points:      {len(df)}")
print(f"Unique trajectories:    {df['trajectory_id'].nunique()}")
print(f"Step count distribution:")
print(df["num_steps"].describe())

df.head(10)

Total data points:      353894
Unique trajectories:    10819
Step count distribution:
count    353894.000000
mean         15.180418
std          16.011929
min           0.000000
25%           6.000000
50%          13.000000
75%          21.000000
max         288.000000
Name: num_steps, dtype: float64


,video_id,trajectory_id,num_steps
0,914-23,914,23
1,3823-25,3823,25
2,7816-14,7816,14
3,8855-11,8855,11
4,10238-17,10238,17
5,8779-15,8779,15
6,9357-9,9357,9
7,6286-4,6286,4
8,7451-17,7451,17
9,7656-1,7656,1


In [4]:
# Cell 4 — Save results
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")

Saved to annotation_trajectory_info.csv


In [ ]:
# Cell 5 — Load motion arrays from cleaned_sorted.json into memory
# Only 3800 trajectories, so loading the full motion dict is safe
import ijson

CLEANED_PATH = "/home/djonna1/scratchtinoosh/iros_dataset/Qwen-Dataset/test/train_r2r_img_motion_cleaned_sorted.json"

motion_map = {}  # {trajectory_id (str): [motion values]}

with open(CLEANED_PATH, "rb") as f:
    for item in ijson.items(f, "item"):
        motion_map[str(item["video_id"])] = item["motion"]

print(f"Loaded motion data for {len(motion_map)} trajectories")

In [6]:
# Cell 6 — Stream-parse annotations.json and build the final dataset
#
# Output format per record:
#   id           : "r2r_{video_id}"
#   image        : [last frame from annotations frames list]
#   gru          : first num_steps values from cleaned_sorted motion array
#   conversations: placeholder format (content left empty for now)

output = []
missing_trajectories = []

with open(ANNOTATIONS_PATH, "rb") as f:
    for item in ijson.items(f, "item"):
        video_id = item["video_id"]
        trajectory_id, num_steps_str = video_id.rsplit("-", 1)
        num_steps = int(num_steps_str)

        # Retrieve motion sequence; warn if trajectory is missing
        if trajectory_id not in motion_map:
            missing_trajectories.append(trajectory_id)
            continue

        motion = motion_map[trajectory_id]
        gru = motion[:num_steps]  # take first num_steps values

        record = {
            "id": f"r2r_{video_id}",
            "image": [item["frames"][-1]],  # last frame only
            "gru": gru,
            "conversations": [
                {"from": "human", "value": ""},
                {"from": "gpt",   "value": ""}
            ]
        }
        output.append(record)

print(f"Built {len(output)} records")
if missing_trajectories:
    print(f"Warning: {len(set(missing_trajectories))} trajectories not found in cleaned_sorted")

# Preview first record
import pprint
pprint.pprint(output[0])

Built 353894 records
{'conversations': [{'from': 'human', 'value': ''},
                   {'from': 'gpt', 'value': ''}],
 'gru': [0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 1, 2, 1],
 'id': 'r2r_914-23',
 'image': ['914/frame_48.jpg']}


In [ ]:
# Cell 7 — Save final dataset
import json

FINAL_OUTPUT_PATH = "/home/djonna1/scratchtinoosh/iros_dataset/Qwen-Dataset/test/r2r_alignment_dataset.json"

with open(FINAL_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"Saved {len(output)} records to {FINAL_OUTPUT_PATH}")

29